# Form Saathi: fine-tune the jargon simplifier (QLoRA)

Teaches a small multilingual model to turn official form language into plain **English, Tamil, Hindi and Odia**.

* Base model: `Qwen/Qwen2.5-1.5B-Instruct` (multilingual, fits a free Colab T4 in 4-bit)
* Method: QLoRA (4-bit NF4 base + LoRA adapters on all attention and MLP projections)
* Loss only on the answer tokens (prompt tokens are masked with -100)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then upload `data/train.jsonl` and `data/val.jsonl`
(built by `python build_dataset.py --augment 400`) using the Files panel on the left.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets

In [ ]:
import json, torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_LEN = 384

def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]

train_rows, val_rows = load("train.jsonl"), load("val.jsonl")
print(len(train_rows), "train /", len(val_rows), "val")

In [ ]:
tok = AutoTokenizer.from_pretrained(BASE)
tok.pad_token = tok.pad_token or tok.eos_token

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

### Tokenise with the chat template and mask the prompt
Only the assistant's plain-language answer contributes to the loss, so the model learns to *write simply* rather than to repeat the jargon.

In [ ]:
def encode(row):
    msgs = row["messages"]
    prompt = tok.apply_chat_template(msgs[:-1], add_generation_prompt=True, tokenize=False)
    full = tok.apply_chat_template(msgs, tokenize=False)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full, add_special_tokens=False)["input_ids"][:MAX_LEN]
    labels = ([-100] * len(p_ids) + f_ids[len(p_ids):])[:len(f_ids)]
    return {"input_ids": f_ids, "attention_mask": [1] * len(f_ids), "labels": labels}

train_ds = Dataset.from_list(train_rows).map(encode, remove_columns=list(train_rows[0].keys()))
val_ds = Dataset.from_list(val_rows).map(encode, remove_columns=list(val_rows[0].keys()))

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, v: seq + [v] * (n - len(seq))
    return {"input_ids": torch.tensor([pad(b["input_ids"], tok.pad_token_id) for b in batch]),
            "attention_mask": torch.tensor([pad(b["attention_mask"], 0) for b in batch]),
            "labels": torch.tensor([pad(b["labels"], -100) for b in batch])}

In [ ]:
args = TrainingArguments(
    output_dir="out", per_device_train_batch_size=8, per_device_eval_batch_size=8,
    gradient_accumulation_steps=2, num_train_epochs=3, learning_rate=2e-4, lr_scheduler_type="cosine",
    warmup_ratio=0.05, fp16=True, logging_steps=10, eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, load_best_model_at_end=True, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
trainer.train()

### Try it

In [ ]:
SYSTEM = train_rows[0]["messages"][0]["content"]
def simplify(text, lang):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"[{lang}] {text}"}]
    prompt = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    enc = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=120, do_sample=False, repetition_penalty=1.1)
    return tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

for lang in ["en", "ta", "hi", "or"]:
    print(lang, "->", simplify("The applicant shall furnish a self-attested copy of the encumbrance certificate.", lang))

### Save the adapter (about 40 MB) and download it
Copy the folder next to `serve_simplifier.py` and run `python serve_simplifier.py`.

In [ ]:
model.save_pretrained("formsaathi-simplifier-lora")
tok.save_pretrained("formsaathi-simplifier-lora")
!zip -qr formsaathi-simplifier-lora.zip formsaathi-simplifier-lora
from google.colab import files
files.download("formsaathi-simplifier-lora.zip")